## 과제 B: 날짜 계산기 (실무 빈도 ★★★★☆)

```python
class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""
    ...
```

---

### **✔️ [0단계] API 키 호출**

In [ ]:
# .env에서 OPENAI_API_KEY 불러오기
import os

from dotenv import load_dotenv
load_dotenv()

# 키 확인
# 다만, 보안을 위해 코드에 명시 금지! 
# print(os.environ.get("OPENAI_API_KEY"))

print("OPENAI_API_KEY 정상 호출" if os.environ.get("OPENAI_API_KEY") else "**키가 확인되지 않습니다. .env 파일을 확인해주세요**")


OPENAI_API_KEY 정상 호출


### **✔️ [0단계] 전역변수 설정**

In [6]:
# 전역변수 설정
from typing import Literal, Optional
# BaseModel : 데이터 형식 검증
# Field : 클래스 필드 description 설명 추가
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
# ToolMessage : (@tool + llm)로 생성한 AI 답변
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
# tool 모듈 : llm이 외부 도구(툴) 사용하게 만드는 모듈
from langchain_core.tools import tool


# 모델 정의
# temperature=0 : 기업에서 자동화 업무 처리를 위해서 -> 명확성, 신뢰성 보장되어야 함
llm = None
try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
except Exception as e:
    print(f"모델 생성 오류: {e}")
    
print("🦾 전역변수 및 모델 세팅 완료" if llm else "실패...❗모듈 이름을 재확인하세요")

🦾 전역변수 및 모델 세팅 완료


### **✔️ [1단계] Pydantic 객체 &nbsp; & &nbsp; 툴(@tool) 정의**

In [57]:
from datetime import datetime

class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """
    날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다.
    
    args = {date1, date2}
    days_diff = data1, date2 두 날짜 간 일수 차이를 출력합니다.
    weeks_diff = data1, date2 두 날짜 간 주수 차이를 출력합니다.
    """
    d1 = datetime.strptime(date1, "%Y-%m-%d")
    d2 = datetime.strptime(date2, "%Y-%m-%d")
    days = abs((d2 - d1).days)
    return DateDiff(date1=date1, date2=date2, days=days, weeks=days // 7)


print("☑️ 툴 days_between 정의 완료!")

☑️ 툴 days_between 정의 완료!


In [ ]:
# 도구 정보 확인
print(f"📛 이름:    {days_between.name}")
print(f"📝 설명:    {days_between.description}")
print(f"🔧 인자:    {days_between.args}")


# llm이 docstring을 읽어온다
'''
args = {date1, date2}
result = data1, date2 두 날짜의 차이를 절댓값으로 출력합니다.
'''

📛 이름:    days_between
📝 설명:    날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다.

args = {date1, date2}
result = data1, date2 두 날짜의 차이를 절댓값으로 출력합니다.
🔧 인자:    {'date1': {'title': 'Date1', 'type': 'string'}, 'date2': {'title': 'Date2', 'type': 'string'}}


### **✔️ [2단계] LLM에 Tool 연결하기 -> bind_tools()**

In [ ]:
# bind_tools() : 앞서 정의한 'llm'에, 툴(days_between) 연결
llm_with_tools_date_calculator = llm.bind_tools([days_between])

# llm_with_tools_date_calculator 자료형(데이터타입) 확인
print(f"타입: {type(llm_with_tools_date_calculator).__name__}")

타입: _ChatModelBinding


### **✔️ [3단계] Tool 연결된 LLM 호출 -> .tool_calls**

In [15]:
# 일반 질문 
# (Tool 없이도 LLM이 대답 가능한 질문)

easy_response = llm_with_tools_date_calculator.invoke("너는 어떤 회사가 만든 LLM이니?")
print(f"💬   Tool 없이   답변: {easy_response.content}")
print(f"🛠️ Tool-Calling 답변: {easy_response.tool_calls}")

💬   Tool 없이   답변: 저는 OpenAI에서 개발한 언어 모델입니다. OpenAI는 인공지능 연구와 개발을 전문으로 하는 회사입니다.
🛠️ Tool-Calling 답변: []


In [17]:
# Tool-calling 필요한 질문
# (days_between 툴을 사용하여 답변할 질문)

date_response = llm_with_tools_date_calculator.invoke("2024년1월1일과 2026년10월8일의 일차와 주차를 계산해줄래?")
print(f"💬   Tool 없이   답변: {date_response.content}")
print(f"🛠️ Tool-Calling 답변: {date_response.tool_calls}")

💬   Tool 없이   답변: 
🛠️ Tool-Calling 답변: [{'name': 'days_between', 'args': {'date1': '2024-01-01', 'date2': '2026-10-08'}, 'id': 'call_CiPQHjMmoyTDp5HvJJ2yO45B', 'type': 'tool_call'}, {'name': 'days_between', 'args': {'date1': '2024-01-01', 'date2': '2026-10-08'}, 'id': 'call_fFcuAMsv4siwoV5X3PPc6fHI', 'type': 'tool_call'}]


In [18]:
tc = date_response.tool_calls[0]   # 첫 번째 도구 호출 정보
print(f"📛 어떤 도구를:  {tc['name']}")
print(f"🔢 어떤 인자로:  {tc['args']}")
print(f"🆔 호출 ID:      {tc['id']}")

📛 어떤 도구를:  days_between
🔢 어떤 인자로:  {'date1': '2024-01-01', 'date2': '2026-10-08'}
🆔 호출 ID:      call_CiPQHjMmoyTDp5HvJJ2yO45B


### **✔️ [4단계] 도구 실행 + 2차 호출(호출 2번)**
### **🟡 LLM 호출 2번(LLM 1차 호출, 2차 호출)**

In [ ]:
# 교안의 Messages와 동일
# 모델의 메모리 만들기
memory = [HumanMessage(content="서기 1년 1월 1일, 2026년 12월 31일의 일 차이, 주 차이를 계산해줘 :)")]

# 1차 호출 - LLM이 도구 호출 의도를 응답

# response = "memory"에 있는 HumanMessage에 대한 툴(days_between) 호출 의사 밝힘
response = llm_with_tools_date_calculator.invoke(memory)
# memory 리스트에 response 누적(저장)
memory.append(response)


print(response)
print(response.content)
print('\n\n')
print(f"🔧 도구 호출 의도: {response.tool_calls}")
print("📜 tool_calls 툴 이름:", response.tool_calls[0]["name"])

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 121, 'total_tokens': 197, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cafde8e9bf', 'id': 'chatcmpl-EWgUAiDRli7vq6nbFvw1y5rmPIE3M', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a11b3f-a13b-7932-aca8-01d398d1e3e9-0' tool_calls=[{'name': 'days_between', 'args': {'date1': '0001-01-01', 'date2': '2026-12-31'}, 'id': 'call_YOrBkEv07Dt8QuhUiUsCYB1c', 'type': 'tool_call'}, {'name': 'days_between', 'args': {'date1': '0001-01-01', 'date2': '2026-12-31'}, 'id': 'call_eVF5gBpxnwsNw4vJVkky9G

In [66]:
### days_between 툴(도구) 실행 ###

memory = memory[:2]   # Human, AI만 남기고 이전에 쌓인 ToolMessage 제거

# 도구 호출마다 실행하고, 호출 ID별로 ToolMessage를 추가
for tc in response.tool_calls:
    result = days_between.invoke(tc["args"])      # DateDiff 객체
    print(f"🗓️ 일수: {result.days}일 | 🗓️ 주수: {result.weeks}주")

    memory.append(ToolMessage(
        content=result.model_dump_json(),
        tool_call_id=tc["id"],
    ))

print(f"✅ 메시지 흐름에 도구 결과 추가됨 (총 {len(memory)}개)")

🗓️ 일수: 739980일 | 🗓️ 주수: 105711주
🗓️ 일수: 739980일 | 🗓️ 주수: 105711주
✅ 메시지 흐름에 도구 결과 추가됨 (총 4개)


### **✔️ 2차 호출 - 최종 답변(자연어) (고객에게 줄 답변)**

In [49]:
# 최종 실행 전,
# memory 리스트 비우기
memory.clear()
print(memory)

[]


In [67]:
# 도구 결과를 본 LLM이 자연어 답변 생성
final = llm_with_tools_date_calculator.invoke(memory)
print(f"💬 최종 답변: {final.content}")

💬 최종 답변: 서기 1년 1월 1일과 2026년 12월 31일 사이의 일수는 **739,980일**이며, 주수는 **105,711주**입니다.


## 과제 C: 가상 날씨 API (실무 빈도 ★★★★☆)

```python
class WeatherInfo(BaseModel):
    city: str
    temperature_c: float
    condition: Literal["맑음", "흐림", "비", "눈"]
    humidity_percent: int

@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다."""
    ...
```

테스트: "서울이랑 부산 날씨 알려줘"   ← 도구를 두 번 호출!
